# Intensity-based refinement of alignment

This recipe will take (bead-based) alignment parameters produced by ```find_transformations_sted.ipynb``` and produce a new set of transformation parameters based on actual image intensity that can be used in ```apply_alignment_to_images.ipynb```.

You need to specify:
* directories containing raw image data (```.msr```) for both the moving and target images
* which channels and which images to include (**Note:** for this to work, both target and moving images should have the same pixel size, e.g. confocal overviews for both)
* path of the original alignment (e.g. bead-based to refine)

At the moment, we only perform translation-based refinement using phase correlation, i.e. the resulting transforms are just shifts. More complex transformation models can most likely be estimated using ```elastix``` but are not implemented yet.



In [42]:
import json
import re
from pathlib import Path
from collections import namedtuple
from itertools import count

import numpy as np
from tqdm import tqdm
from scipy.spatial import KDTree

from msr_reader import OBFFile
from calmutils.stitching import get_axes_aligned_overlap,translation_matrix
from calmutils.stitching.fusion import fuse_image
from calmutils.stitching.phase_correlation import phasecorr_align
from transform_helpers import world_transform_to_pixel_transform
from transform_helpers import get_scan_field_metadata, world_coords_for_pixel_spots


# convenience namedtuple for pre-loaded image data for moving / transformed images
MovingImageData = namedtuple('MovingImageData', ['imgs', 'transform', 'coord_origin', 'coord_center', 'pixel_size'])


In [43]:
base_path_target = '/Volumes/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240803_DNAFISH/'
base_path_moving = '/Volumes/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240802_RNAFISH/'

msr_subdir_target = 'raw'
msr_subdir_moving = 'raw'

alignment_params_subdir = 'alignment_parameters'
alignment_params_file_existing = 'alignment_parameters_local.json'
alignment_params_file_out = 'alignment_parameters_intensity_based.json'

# include and exclude patterns for files
# exclude: do not process files containing this pattern, include: only process files containing a pattern
# can be used to e.g. only process overview/sted images.
# NOTE: this only really makes sense for overviews, so we exclude 'sted' files by default
file_exclude_pattern_target = 'sted'
file_include_pattern_target = None
file_exclude_pattern_moving = 'sted'
file_include_pattern_moving = None

# channels to include in alignment, if multiple are specified, they will be averaged
channels_to_include_target = (0, 1)
channels_to_include_moving = (0,)

# alignment parameters
# image intensities are clipped at a high quantile to avoid influence of small bright structures, e.g. FISH spots
# 0.98 - 0.99 seems to work well
clip_quantile = 0.985
# minimal cross correlation of registered images to keep. > 0.3 has few false positives
min_cross_corr = 0.3


## 1) Load existing transformation field

In [44]:
with open(Path(base_path_moving) / alignment_params_subdir / alignment_params_file_existing) as fd:
    parameters = json.load(fd)

# get center coords (in moving image world coordinates), build kd-tree for quick nearest lookup
center_coords_moving = [np.array(p['center_coords']) for p in parameters['transformations']]
kd_transform_centers = KDTree(center_coords_moving)

# get transformation matrix entries in same order, back to 4x4
transformations_moving = [np.array(p['parameters']).reshape((4,4)) for p in parameters['transformations']]

## 2) Load moving images

We now load all moving images and get their transformations plus some metadata

In [ ]:
moving_imgs_data = []
for msr_file_moving in tqdm(list((Path(base_path_moving) / msr_subdir_moving).glob('*.msr'))):

    # check if filename doesn't match an include pattern or if it does match an exclude pattern -> skip 
    if file_include_pattern_moving is not None and not re.findall(file_include_pattern_moving, msr_file_moving.name):
        continue
    if file_exclude_pattern_moving is not None and re.findall(file_exclude_pattern_moving, msr_file_moving.name):
        continue

    with OBFFile(msr_file_moving) as reader:
        imgs = [reader.read_stack(i) for i in channels_to_include_moving]
    
    meta = get_scan_field_metadata(msr_file_moving, channels_to_include_moving[0])
    shape = np.array(imgs[0].shape)
    coord_origin = world_coords_for_pixel_spots([0,0,0], meta)[0] * 1e6
    coord_center = world_coords_for_pixel_spots(shape/2, meta)[0] * 1e6

    # find closest transform in transformation field
    _, closest_transform_idx = kd_transform_centers.query(coord_center)
    transform = transformations_moving[closest_transform_idx]

    # save images, (world coord) transform, origin, center and pixel size
    img_data = MovingImageData(imgs, transform, coord_origin, coord_center, meta.pixel_size * 1e6)
    moving_imgs_data.append(img_data)


## 3) Go over Target Images, match to moving, find transform

Now, we will go through all target images, and for each of them select overlapping moving images based on their existing transforms.

For all matching pairs, we align using phase correlation and prepare a list of transformation parameters similar to the bead-based notebook.

In [ ]:
results = {'transformations': []}

for msr_file_target in tqdm(list((Path(base_path_target) / msr_subdir_target).glob('*.msr'))):

    # check if filename doesn't match an include pattern or if it does match an exclude pattern -> skip 
    if file_include_pattern_target is not None and not re.findall(file_include_pattern_target, msr_file_target.name):
        continue
    if file_exclude_pattern_target is not None and re.findall(file_exclude_pattern_target, msr_file_target.name):
        continue

    # read channels to include in result
    with OBFFile(msr_file_target) as reader:
        imgs = [reader.read_stack(i) for i in channels_to_include_target]

    # get image metadata
    meta = get_scan_field_metadata(msr_file_target, channels_to_include_target[0])
    shape = np.array(imgs[0].shape)
    coord_origin = world_coords_for_pixel_spots([0,0,0], meta)[0] * 1e6
    coord_center = world_coords_for_pixel_spots(shape/2, meta)[0] * 1e6

    # average and clip target images
    avg_target = np.mean(imgs, axis=0)
    avg_target_clip = np.clip(avg_target, 0, np.quantile(avg_target, clip_quantile))

    for i, moving_img_data in enumerate(moving_imgs_data):

        # get transform in pixel units
        transform_i_moving = world_transform_to_pixel_transform(moving_img_data.transform, coord_origin, moving_img_data.coord_origin, meta.pixel_size * 1e6, moving_img_data.pixel_size)
        # check overlap (of axis-aligned transformed image)
        mins, maxs = get_axes_aligned_overlap(imgs[0].shape, moving_img_data.imgs[0].shape, None, transform_i_moving)

        # there is some overlap
        if not all(mins < maxs):
            continue

        avg_moving = np.mean(moving_img_data.imgs, axis=0)
        avg_moving_clip = np.clip(avg_moving, 0, np.quantile(avg_moving, clip_quantile))

        shift, corr = phasecorr_align(avg_target_clip, avg_moving_clip)

        if corr > min_cross_corr:
            # final translation: centermov-centertarget + shift * pixelsize
            world_shift = np.array(shift) * meta.pixel_size * 1e6 - (moving_img_data.coord_origin - coord_origin)
            transform_info = {
                'center_coords': list(map(float, moving_img_data.coord_center)),
                'corr': corr,
                'transform_type': 'translation',
                'parameters': list(map(float, translation_matrix(world_shift).flat))
            }

            results['transformations'].append(transform_info)


## 4) Save Transformation Parameters

In [52]:
with open(Path(base_path_moving) / alignment_params_subdir / alignment_params_file_out, 'w') as fd:
    json.dump(results, fd, indent=1)